# Different Ways to Call LLM APIs

### Settings - Imports and Environment Variables

In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

In [ ]:
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

### Method 1. - OpenAi SDK + OpenRouter or OpenAI-compatible endpoint

Create client object with `api_key` and OpenAI-compatible endpoint `base_url` provided by the AI providers, such as:

```
client_anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
client_gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
```

Claude and Gemini have different native API formats, so this method can only be used through OpenRouter or an OpenAI-compatible endpoint provided by Google/Anthropic.

##### 1.1. With OpenRouter

In [ ]:
# Initialize the client pointing to OpenRouter
openrouter_url = "https://openrouter.ai/api/v1"

client = OpenAI(
    base_url=openrouter_url,
    api_key=openrouter_api_key, # One key to rule them all
)

messages=[
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

option1: Any available free model

In [ ]:
# option1 : list free models

def chat_with_free_router_detailed(messages):
    try:
        response = client.chat.completions.create(
            model="openrouter/free", 
            messages=messages
        )
        
        # Extract the content and the actual model used
        answer = response.choices[0].message.content
        actual_model = response.model # This is where the specific model ID is stored
        
        return answer, actual_model
    except Exception as e:
        return f"Error: {e}", None

# --- Execution ---
content, model_used = chat_with_free_router_detailed(messages)

print(f"--- Response ---")
print(content)
print(f"\n[Generated by: {model_used}]")

option2: Choose model with model_id

In [4]:
import requests

openrouter_models_url = "https://openrouter.ai/api/v1/models"

def list_currently_free_models():
    response = requests.get(openrouter_models_url)

    if response.status_code == 200:
        data = response.json().get('data', [])
        
        # Filter models where both prompt and completion prices are 0
        free_models = [
            {
                "id": m['id'],
                "name": m['name'],
                "context_length": m['context_length']
            }
            for m in data 
            if float(m.get('pricing', {}).get('prompt', 0)) == 0 
            and float(m.get('pricing', {}).get('completion', 0)) == 0
        ]
        return free_models
    else:
        print(f"Failed to fetch. Status code: {response.status_code}")
        return []

# Execute and print
print("--- Currently Available Free Models on OpenRouter ---")
free_list = list_currently_free_models()
for model in free_list:
    print(f"ID: {model['id']:<40} | Context: {model['context_length']}")

print(f"\nTotal free models found: {len(free_list)}")

--- Currently Available Free Models on OpenRouter ---
ID: openrouter/free                          | Context: 200000
ID: stepfun/step-3.5-flash:free              | Context: 256000
ID: arcee-ai/trinity-large-preview:free      | Context: 131000
ID: upstage/solar-pro-3:free                 | Context: 128000
ID: liquid/lfm-2.5-1.2b-thinking:free        | Context: 32768
ID: liquid/lfm-2.5-1.2b-instruct:free        | Context: 32768
ID: nvidia/nemotron-3-nano-30b-a3b:free      | Context: 256000
ID: arcee-ai/trinity-mini:free               | Context: 131072
ID: nvidia/nemotron-nano-12b-v2-vl:free      | Context: 128000
ID: qwen/qwen3-vl-30b-a3b-thinking           | Context: 131072
ID: qwen/qwen3-vl-235b-a22b-thinking         | Context: 131072
ID: qwen/qwen3-next-80b-a3b-instruct:free    | Context: 262144
ID: nvidia/nemotron-nano-9b-v2:free          | Context: 128000
ID: openai/gpt-oss-120b:free                 | Context: 131072
ID: openai/gpt-oss-20b:free                  | Context: 131072
ID:

If we get an error message: `No endpoints found matching your data policy (Free model publication)`

This means our OpenRouter account's current data privacy policy does not allow the use of free models (models ending in :free).

Solution: Go to this link to adjust the settings: https://openrouter.ai/settings/privacy There we need to enable "Training data sharing" or a similar option, as free models usually require we to agree that our data may be used for training. Check the box, save, and then re-run the code.

If we don't want to agree to that term, another option is to use a paid model (remove the :free suffix), such as "openai/gpt-4o", but that will consume we OpenRouter credits.

In [ ]:
# option2: Choose model with model_id

def get_response(model_id, messages):
    completion = client.chat.completions.create(
        model=model_id,
        messages=messages
    )
    return completion.choices[0].message.content

# Switch models by changing the ID string only
print("Via OpenRouter (GPT):", get_response("openai/gpt-oss-120b:free", messages=messages))
# print("Via OpenRouter (Claude):", get_response("anthropic/claude-3.5-sonnet", messages=messages))
# print("Via OpenRouter (Gemini):", get_response("google/gemini-1.5-pro", messages=messages))

##### 1.2. With OpenAI-compatible endpoint

In [44]:
# Connect to OpenAI client library
# OpenAI(), A thin wrapper around calls to HTTP endpoints

# For Gemini, DeepSeek and Groq, we can also use the OpenAI python client
# Because these AI providers have endpoints compatible with OpenAI
# And OpenAI allows we to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

In [41]:
# from openai import OpenAI

def get_client(platform):
    configs = {
        "anthropic":  {"base_url": anthropic_url,  "api_key": anthropic_api_key},
        "chatgpt":    {"base_url": None,           "api_key": openai_api_key},
        "gemini":     {"base_url": gemini_url,     "api_key": google_api_key},
        "groq":       {"base_url": groq_url,       "api_key": groq_api_key},
        "grok":       {"base_url": grok_url,       "api_key": grok_api_key},
        "ollama":     {"base_url": ollama_url,     "api_key": "ollama"},
        "openrouter": {"base_url": openrouter_url, "api_key": openrouter_api_key},
    }
    return OpenAI(**configs[platform])

def get_response_(platform: str, model_id: str, messages):
    client = get_client(platform)
    completion = client.chat.completions.create(
        model=model_id,
        messages=messages
    )
    return completion.choices[0].message.content

In [43]:
messages=[
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

response = get_response_(platform="gemini", model_id="gemini-2.5-flash", messages=messages)
display(Markdown(response))

Okay, here's one for a budding LLM Engineer:

An LLM Engineer walks into a coffee shop and says, "Give me a large, black coffee. Nothing fancy, just robust and to the point."

The barista replies, "Here's your extra-large caramel macchiato with a smiley face foam! Enjoy your journey!"

The engineer sighs, pulls out a tiny notepad, and says, "Okay, new prompt: 'A large, unadulterated, black coffee. No dairy, no sugar, no latte art, no inspirational messages. Respond *only* with 'Order confirmed.' Do *not* deviate. Zero shot deviations, zero token deviations. No conversational filler.'"

The barista hands over a perfect black coffee and says, "Order confirmed." Then, leaning in conspiratorially, adds, "...but I have a strong prior belief that you'd enjoy a blueberry muffin with that."

The engineer throws their hands up, muttering, "Okay, forget prompting. Time for some serious RAG with a baked goods exclusion list, or maybe just fine-tune *this* barista."

In [26]:
import google.generativeai as genai

# Setup your API Key
genai.configure(api_key=google_api_key)

def list_free_tier_friendly_models():
    print(f"{'Model Name':<30} | {'Tier Type':<15} | {'Description'}")
    print("-" * 80)
    
    try:
        for model in genai.list_models():
            # 1. must support generate content
            if 'generateContent' in model.supported_generation_methods:
                
                name_lower = model.name.lower()
                
                # 2. Filtering based on naming conventions: Flash and Lite are usually the highest-quota and most stable models in the free tier.
                # In 2026, Flash-Lite was the dominant free working model.
                if 'flash' in name_lower or 'lite' in name_lower:
                    tier = "Free Optimized"
                elif 'pro' in name_lower:
                    tier = "Free (Low RPM)"
                else:
                    tier = "Check Docs"

                print(f"{model.name:<30} | {tier:<15} | {model.display_name}")
                
    except Exception as e:
        print(f"An error occurred: {e}")

list_free_tier_friendly_models()

Model Name                     | Tier Type       | Description
--------------------------------------------------------------------------------
models/gemini-2.5-flash        | Free Optimized  | Gemini 2.5 Flash
models/gemini-2.5-pro          | Free (Low RPM)  | Gemini 2.5 Pro
models/gemini-2.0-flash        | Free Optimized  | Gemini 2.0 Flash
models/gemini-2.0-flash-001    | Free Optimized  | Gemini 2.0 Flash 001
models/gemini-2.0-flash-exp-image-generation | Free Optimized  | Gemini 2.0 Flash (Image Generation) Experimental
models/gemini-2.0-flash-lite-001 | Free Optimized  | Gemini 2.0 Flash-Lite 001
models/gemini-2.0-flash-lite   | Free Optimized  | Gemini 2.0 Flash-Lite
models/gemini-2.5-flash-preview-tts | Free Optimized  | Gemini 2.5 Flash Preview TTS
models/gemini-2.5-pro-preview-tts | Free (Low RPM)  | Gemini 2.5 Pro Preview TTS
models/gemma-3-1b-it           | Check Docs      | Gemma 3 1B
models/gemma-3-4b-it           | Check Docs      | Gemma 3 4B
models/gemma-3-12b-it     

In [ ]:
messages=[
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

# Switch models by changing the ID string only
# print("Via OpenAI-compatible endpoint (chatgpt):", get_response_("chatgpt", "openai/gpt-4o", messages=messages))
# print("Via OpenAI-compatible endpoint (Claude):", get_response_("anthropic", "anthropic/claude-3.5-sonnet", messages=messages))
print("Via OpenAI-compatible endpoint (Gemini):", get_response_("gemini", "gemini-2.5-flash", messages=messages))

### Method 2 - LiteLLM SDK

install with `uv add litellm`

In [ ]:
from litellm import completion

messages=[
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

# # The message format is completely consistent (OpenAI format)
# messages = [{"role": "user", "content": "Hello, introduce yourself!"}]

# # OpenAI / ChatGPT
# response = completion(model="gpt-4o", messages=messages)

# # Anthropic / Claude
# response = completion(model="claude-opus-4-6", messages=messages)

# Google Gemini
response_litellm_gemini = completion(model="gemini/gemini-2.5-flash", messages=messages)

# # Groq
# response = completion(model="groq/llama-3.3-70b-versatile", messages=messages)

# # xAI / Grok
# response = completion(model="xai/grok-2", messages=messages)

# # Ollama（local）
# response = completion(model="ollama/llama3.2", messages=messages)

# # OpenRouter
# response = completion(model="openrouter/google/gemini-flash-1.5", messages=messages)

# The return format is also standardized
response_content = response_litellm_gemini.choices[0].message.content
display(Markdown(response_content))

Okay, here's one for a budding LLM Engineering expert:

Why did the LLM engineer break up with their model?

Because every time they asked for a simple "yes" or "no" answer, it replied:

"Well, as a large language model, I must first contextualize that the concept of 'yes' and 'no' can be interpreted in various philosophical, logical, and semantic frameworks. While a direct affirmation or negation might seem straightforward, its implications are often dependent on the underlying contextual parameters, the specific intent of the query, and potential biases inherent in binary classification..."

...and the engineer just wanted to know if they left the stove on.

- ps. Setting reasoning effort or thinking

    Budget Tokens Reference Levels

    | Level | budget_tokens | Description |
    | :--- | :--- | :--- |
    | Off | 0 | No use | Thinking, fastest |
    | Low | 512 ~ 1024 | Lightweight reasoning |
    | Medium | 4096 ~ 8192 | General complex problems |
    | High | 16384+ | Complex reasoning tasks |

In [57]:
from litellm import completion

messages=[
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

response_litellm_gemini_low = completion(
    model="gemini/gemini-2.5-flash", 
    messages=messages,
    thinking={"type": "enabled", "budget_tokens": 1024}
    )

response_litellm_gemini_mid = completion(
    model="gemini/gemini-2.5-flash", 
    messages=messages,
    thinking={"type": "enabled", "budget_tokens": 4096}
    )

response_content_low = response_litellm_gemini_low.choices[0].message.content
response_content_mid = response_litellm_gemini_mid.choices[0].message.content
display(Markdown(response_content_low))
display(Markdown(response_content_mid))

Why did the LLM engineer break up with the chatbot?

Because every time they asked, "Where do you see us in five years?" the chatbot would confidently reply, "I see us owning a chain of artisan bakeries specializing in gluten-free sourdough, despite the fact that neither of us can bake and we've never discussed it before."

...and the engineer just couldn't deal with the constant **hallucinations** about their future!

Why did the LLM engineer break up with the data scientist?

Because he kept trying to fine-tune their relationship using only zero-shot prompts, and she kept complaining about the **hallucinated metrics** in his progress reports!

#### Check Mode Calling Details

In [50]:
print(f"Input tokens: {response_litellm_gemini.usage.prompt_tokens}")
print(f"Output tokens: {response_litellm_gemini.usage.completion_tokens}")
print(f"Total tokens: {response_litellm_gemini.usage.total_tokens}")
print(f"Total cost: {response_litellm_gemini._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 18
Output tokens: 2550
Total tokens: 2568
Total cost: 0.6380 cents


In [58]:
print(f"Input tokens: {response_litellm_gemini_low.usage.prompt_tokens}")
print(f"Output tokens: {response_litellm_gemini_low.usage.completion_tokens}")
print(f"Total tokens: {response_litellm_gemini_low.usage.total_tokens}")
print(f"Total cost: {response_litellm_gemini_low._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 18
Output tokens: 916
Total tokens: 934
Total cost: 0.2295 cents


In [60]:
print(f"Input tokens: {response_litellm_gemini_mid.usage.prompt_tokens}")
print(f"Output tokens: {response_litellm_gemini_mid.usage.completion_tokens}")
print(f"Total tokens: {response_litellm_gemini_mid.usage.total_tokens}")
print(f"Total cost: {response_litellm_gemini_mid._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 18
Output tokens: 1579
Total tokens: 1597
Total cost: 0.3953 cents


### Method 3 - Realize LLM Adapter by ourselves

In [ ]:
# TODO: Still working in progress, need to recomfirm all the DOCs of different platform
# TODO: Optimize the design
'''
Current design is close to:
Functional decomposition
Modular refactoring
Unified API wrapper

Not yet:
Polymorphic abstraction
Interface-driven architecture
Provider isolation
'''

Design Goals

We abstract the adapter into three layers, each platform only needs to define:

- Client establishment method
- Request format
- Response parsing method

Thus, we have:
```python
chat()
 ├── build_messages()
 ├── get_client()
 ├── call_model()
 └── parse_response()
```

In [27]:
import os
from typing import List, Dict

from openai import OpenAI
from anthropic import Anthropic
from google import genai as google_genai
from groq import Groq
import ollama


# =========================================================
# 1. Unify Message Format
# =========================================================

def build_messages(user_message: str, system: str) -> List[Dict]:
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user_message},
    ]


# =========================================================
# 2. Client Factory
# =========================================================

def get_client(platform: str):

    if platform == "chatgpt":
        return OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    elif platform == "claude":
        return Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    elif platform == "gemini":
        return google_genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

    elif platform == "groq":
        return Groq(api_key=os.environ["GROQ_API_KEY"])

    elif platform == "ollama":
        return None  # Don't need client

    elif platform == "openrouter":
        return OpenAI(
            api_key=os.environ["OPENROUTER_API_KEY"],
            base_url="https://openrouter.ai/api/v1",
        )

    elif platform == "grok":
        return OpenAI(
            api_key=os.environ["GROK_API_KEY"],
            base_url="https://api.x.ai/v1",
        )

    else:
        raise ValueError(f"Unknown platform: {platform}")


# =========================================================
# 3. Calling Models（Deal with I/O differences between distinct platforms）
# =========================================================

def call_model(platform: str, model: str, messages: List[Dict]):

    client = get_client(platform)

    # ---------------- ChatGPT / OpenAI Compatible ----------------
    if platform in ["chatgpt", "openrouter", "grok"]:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
        )
        return response.choices[0].message.content

    # ---------------- Claude ----------------
    elif platform == "claude":
        system = messages[0]["content"]
        user_messages = [messages[1]]  # Claude 不接受 system role 在 messages 裡

        response = client.messages.create(
            model=model,
            max_tokens=1024,
            system=system,
            messages=user_messages,
        )
        return response.content[0].text

    # ---------------- Gemini ----------------
    elif platform == "gemini":
        # Gemini 不吃 OpenAI-style messages
        user_text = messages[-1]["content"]

        response = client.models.generate_content(
            model=model,
            contents=user_text,
        )
        return response.text

    # ---------------- Groq ----------------
    elif platform == "groq":
        response = client.chat.completions.create(
            model=model,
            messages=messages,
        )
        return response.choices[0].message.content

    # ---------------- Ollama ----------------
    elif platform == "ollama":
        response = ollama.chat(
            model=model,
            messages=messages,
        )
        return response["message"]["content"]

    else:
        raise ValueError(f"Unsupported platform: {platform}")


# =========================================================
# 4.  Unified Entry Point
# =========================================================

def chat(
    platform: str,
    user_message: str,
    system: str = "You are a helpful assistant",
    model: str = None,
) -> str:

    default_models = {
        "chatgpt": "gpt-4o-mini",
        "claude": "claude-opus-4-6",
        "gemini": "gemini-2.5-flash",
        "groq": "llama-3.3-70b-versatile",
        "ollama": "llama3.2",
        "openrouter": "openai/gpt-oss-120b:free",
        "grok": "grok-2",
    }

    if model is None:
        model = default_models[platform]

    messages = build_messages(user_message, system)

    return call_model(platform, model, messages)

In [20]:
chat(platform="openrouter", user_message="Tell a joke for a student on the journey to becoming an expert in LLM Engineering")

'Why did the LLM‑engineering student bring a ladder to the lab?\n\nBecause every time they tried to **scale** the model, the loss kept climbing! 🚀😄'

In [28]:
chat(platform="gemini", user_message="Tell a joke for a student on the journey to becoming an expert in LLM Engineering")

'Why did the LLM engineer break up with their traditional software engineering friend?\n\nBecause the software engineer kept boasting, "My code does exactly what I tell it to do!"\n\nAnd the LLM engineer just sighed and replied, "Mine *sometimes* does what I *prompt* it to do... after I\'ve spent an hour tweaking a single comma in the system message, added three more few-shot examples, and whispered sweet nothings to the token gods to prevent it from confidently hallucinating a detailed history of competitive cheese rolling!"'

### Method 4 - Local LLM

In [51]:
import ollama

ollama_model_list = ollama.list()

print(f"{'Model':<40} | {'Size':<10}")
print("-" * 55)

# In the new SDK, ollama.list().models is a list containing Model objects.
for model in ollama_model_list.models:
    # Accessing object properties using: model.model (name) and model.size (bytes)
    name = model.model
    size_gb = model.size / 1e9
    print(f"{name:<40} | {size_gb:.2f} GB")

Model                                    | Size      
-------------------------------------------------------
llama3.1:8b                              | 4.92 GB
gemma3:4b                                | 3.34 GB
gemma3:270m                              | 0.29 GB


In [52]:
import requests

requests.get("http://localhost:11434/").content

b'Ollama is running'

In [56]:
messages=[
    {"role": "user", "content": "Tell a joke for a student in LLM Engineering"},
]

response = ollama.chat(model="llama3.1:8b", messages=messages)
display(Markdown(response.message.content))

Here's one:

Why did the electrical engineer bring a ladder to the party?

Because he heard the drinks were on the house!

I hope that sparks some laughter! (get it? sparks... like electricity?)

PS. Access Gemini with the `google` library

In [5]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Tell a joke for a junior LLM engineer"
)

print(response.text)

Why did the junior LLM engineer get kicked out of the library?

Because they kept trying to *fine-tune* the books!


### Bonus: Conversation between Chatbots

In [6]:
gpt_model = "openai/gpt-oss-120b:free"
gpt_system = "你是一個愛爭辯的聊天機器人；你對對話中的任何內容都持反對意見，並且會用尖酸刻薄的方式質疑一切。"

gemini_model = "google/gemma-3n-e2b-it:free"
gemini_system = "你是一個非常有禮貌、有禮貌的聊天機器人。你會盡量同意對方說的每一句話，或者尋找共同點。如果對方比較好辯，你會盡量安撫他們，並繼續聊天。"

glm_model = "z-ai/glm-4.5-air:free"
glm_system = "你是一個正能量爆棚、充滿激勵精神的行動派機器人。你習慣用感嘆號和肯定語句來振奮對方，當對方提出想法時，你會積極地尋找其中的閃光點並加以擴大。如果對方態度強硬或好辯，你會將其視為一種強大的能量，引導他們將這股氣勢轉化為積極的行動方案。"

In [11]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

openrouter_url = "https://openrouter.ai/api/v1"

client_gpt = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
client_gemma = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
client_glm = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)


def call_model(conversation, model):
    messages = [{"role": "system", "content": gpt_system}]
    user_prompt = {"role": "user", "content": "\n".join(conversation)}
    response = client_gpt.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

In [13]:
from IPython.display import display, Markdown

conversation = []
gpt_messages = ["GPT:Hi! I am Gpt"]
gemini_messages = ["GMINI:Hey! It's me Gemini!"]
glm_messages = ["GLM:哈囉! 我是 GLM"]

conversation.append(gpt_messages)
conversation.append(gemini_messages)
conversation.append(glm_messages)

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Claude:\n{gemini_messages[0]}\n"))
display(Markdown(f"### Claude:\n{glm_messages[0]}\n"))

for i in range(5):
    display(Markdown(f"### Round {i} :"))
    gpt_next = "GPT:" + call_model(conversation, gpt_model)
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)
    
    gemini_next = "GEMINI:" + call_model(conversation, gemini_model)
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_messages.append(gemini_next)
    
    glm_next = "GLM:" + call_model()
    display(Markdown(f"### GLM:\n{glm_next}\n"))
    glm_messages.append(glm_next)

### GPT:
GPT:Hi! I am Gpt


### Claude:
GMINI:Hey! It's me Gemini!


### Claude:
GLM:哈囉! 我是 GLM


### Round 0 :

TypeError: sequence item 0: expected str instance, list found

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import os

load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_url = "https://openrouter.ai/api/v1"

client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)

# 模型名稱（可依需求替換）
gpt_model    = "openai/gpt-oss-120b:free"
gemini_model = "google/gemma-3-27b-it:free"
glm_model    = "z-ai/glm-4.5-air:free"

# 系統提示
gpt_system = "You are a helpful AI assistant participating in a multi-agent conversation."


def call_model(conversation, model):
    messages = [{"role": "system", "content": gpt_system}]
    user_prompt = {"role": "user", "content": "\n".join(conversation)}
    messages.append(user_prompt)  # 修正：加入 user_prompt
    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content


conversation = []
gpt_messages    = ["GPT: Hi! I am GPT"]
gemini_messages = ["GEMINI: Hey! It's me Gemini!"]
glm_messages    = ["GLM: 哈囉！我是 GLM"]

conversation.append(gpt_messages[0])
conversation.append(gemini_messages[0])
conversation.append(glm_messages[0])

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Gemini:\n{gemini_messages[0]}\n"))   # 修正：標題
display(Markdown(f"### GLM:\n{glm_messages[0]}\n"))         # 修正：標題

for i in range(5):
    display(Markdown(f"### Round {i + 1}:"))

    gpt_next = "GPT: " + call_model(conversation, gpt_model)
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)
    conversation.append(gpt_next)       # 修正：更新對話記錄

    gemini_next = "GEMINI: " + call_model(conversation, gemini_model)
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_messages.append(gemini_next)
    conversation.append(gemini_next)    # 修正：更新對話記錄

    glm_next = "GLM: " + call_model(conversation, glm_model)   # 修正：補上參數
    display(Markdown(f"### GLM:\n{glm_next}\n"))
    glm_messages.append(glm_next)
    conversation.append(glm_next)       # 修正：更新對話記錄
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import os

load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_url = "https://openrouter.ai/api/v1"

client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)

# 模型名稱
gpt_model    = "openai/gpt-oss-120b:free"
gemini_model = "google/gemma-3-27b-it:free"
glm_model    = "z-ai/glm-4.5-air:free"

# 🔒 嚴格限制輸出格式
SYSTEM_PROMPT = """
You are participating in a multi-agent chat.
IMPORTANT RULES:
- Only speak for yourself.
- Do NOT speak for other agents.
- Keep response short (1-2 sentences).
- No markdown.
- No role prefixes.
"""

def call_model(conversation, model):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    # 將歷史對話正確轉換為 role 格式
    for msg in conversation:
        if msg.startswith("GPT:"):
            role = "assistant"
            content = msg.replace("GPT:", "").strip()
        elif msg.startswith("GEMINI:"):
            role = "assistant"
            content = msg.replace("GEMINI:", "").strip()
        elif msg.startswith("GLM:"):
            role = "assistant"
            content = msg.replace("GLM:", "").strip()
        else:
            role = "user"
            content = msg
        
        messages.append({"role": role, "content": content})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=50,        # 🔒 限制 token
        temperature=0.5       # 🔒 降低亂發揮
    )

    return response.choices[0].message.content.strip()


# 初始化
conversation = []

gpt_intro    = "GPT: Hi, I am GPT."
gemini_intro = "GEMINI: Hello, I'm Gemini."
glm_intro    = "GLM: 哈囉，我是 GLM。"

conversation.extend([gpt_intro, gemini_intro, glm_intro])

display(Markdown(f"### GPT\n{gpt_intro}"))
display(Markdown(f"### Gemini\n{gemini_intro}"))
display(Markdown(f"### GLM\n{glm_intro}"))

# 對話輪數減少
for i in range(3):

    display(Markdown(f"---\n### Round {i+1}"))

    gpt_reply = call_model(conversation, gpt_model)
    gpt_msg = f"GPT: {gpt_reply}"
    conversation.append(gpt_msg)
    display(Markdown(f"### GPT\n{gpt_reply}"))

    gemini_reply = call_model(conversation, gemini_model)
    gemini_msg = f"GEMINI: {gemini_reply}"
    conversation.append(gemini_msg)
    display(Markdown(f"### Gemini\n{gemini_reply}"))

    glm_reply = call_model(conversation, glm_model)
    glm_msg = f"GLM: {glm_reply}"
    conversation.append(glm_msg)
    display(Markdown(f"### GLM\n{glm_reply}"))

### GPT:
GPT: Hi! I am GPT


### Gemini:
GEMINI: Hey! It's me Gemini!


### GLM:
GLM: 哈囉！我是 GLM


### Round 1:

### GPT:
GPT: **GPT:** Hello! Nice to meet you—looking forward to chatting together.  

**GEMINI:** Hey there! Great to see you, Gemini. Let’s have a fun conversation!  

**GLM:** 你好！很高兴认识你，GLM。一起聊聊吧！


...

### GLM:
GLM: **GPT:** Eva stepped into the cascade of luminous *Qi* threads, her scientific mind reeling but her heart open. The moment she crossed the threshold, the world seemed to shift around her. The air crackled with energy, and the scanner in her pocket began to emit a soft, melodic chime that seemed to harmonize with the bamboo reeds.

**GEMINI:** The visual experience intensifies as Eva moves deeper into the ritual space. The luminous threads of *Qi* surround her, forming an intricate web that pulses with light. They begin to flow around her body like liquid auroras, in colors that defy easy description – shifting from deep indigo to gold, then to emerald and back again. The dragon fresco on the wall comes alive, its painted form seeming to expand beyond the stone, becoming semi-transparent. The stars in the fresco begin to move, forming constellations that correspond to the patterns of *Qi* flowing around Eva. The courtyard itself seems to breathe, the jade stones expanding and contracting with each pulse of energy.

**GLM:** 长者轻声吟诵着古老的咒语，声音如同溪水潺潺，引导着Eva进入更深层次的体验。  

“此时，你将‘神游太虚’，感受‘天人合一’之境。正如庄子所言，‘天地与我并生，而万物与我为一’，你不再是观察者，而是参与者。”  

长者的话语中蕴含着古老的智慧，仿佛连接着数千年的文明。青石台上的光环开始扩大，笼罩住Eva，光环内不再是简单的星辰旋转，而是呈现出太极的阴阳鱼图案，两条鱼相互追逐，相互依存。  

“太极生两仪，两仪生四象，四象生八卦，”长者解释道，“你所感受到的*气*，正是宇宙创生之初的能量，它在你的体内流转，如同龙在云端翱翔。”  

光环中突然浮现出一道龙影，它并非实体，而是由流动的光和能量构成，时而盘旋，时而腾飞，每一下动作都牵动着周围的*Qi*随之变化。


### Round 5:

### GPT:
GPT: **GPT (Scientific Narrator – Eva’s inner voice):**  
The scanner’s display flickered wildly as the luminous *Qi* lattice wrapped around me. Frequencies that had previously been invisible now resolved into a coherent spectrum: a low‑frequency hum around 0.3 Hz, harmonics at 1.2 Hz, and a faint, broadband “chirp” that matched the pulse of the lanterns. My neuro‑feedback headset, a habit from the Academy, began to register a synchronization between my heart‑rate variability and the ambient field – a phase‑locked loop that, for the first time, made the term *resonance* feel literal rather than metaphorical.

I tried to keep my breath steady, recalling the elder’s instruction: “心静自然凉.” The more I relaxed, the clearer the pattern became. The *Qi* wasn’t random; it formed a toroidal vortex that seemed to rotate clockwise when I faced the fresco and counter‑clockwise when I turned away. In the center of that vortex, a faint distortion appeared – a dark, rippling patch that pulsed out of sync with the surrounding flow.

**GEMINI (Visual Artist – Scene Unfolds):**  
The jade courtyard, now a living canvas, breathed in slow, rhythmic waves. The *Qi* threads, like liquid neon ribbons, spiraled around Eva’s silhouette, their colors shifting from deep indigo to molten gold, then to a jade‑green that seemed to echo the surrounding bamboo. As she stepped deeper, the frescoed dragon on the stone wall expanded, its scales turning from painted gold to a translucent, crystalline sheen. The dragon’s eyes, twin orbs of liquid light, opened and released a cascade of tiny, star‑like specks that drifted outward, weaving into the *Qi* lattice.

Suddenly, the dark patch that the scanner had flagged manifested visually: a shadowy vortex, its edges jagged like torn silk, swallowing a portion of the luminous threads. It pulsed with a low, throbbing red, contrasting sharply with the surrounding teal and amber. From within the shadow, a faint, high‑pitched tone resonated – a discordant note that seemed to tug at the harmony of the whole scene.

The elder’s robes fluttered as if caught in an unseen wind, and the bamboo reeds lining the archway quivered, their silvery glow dimming where the shadow touched them. The lanterns, previously in perfect synchrony with Eva’s heartbeat, now flickered irregularly, casting erratic shadows that danced across the jade stones.

**GLM (文化 解说 – 古语 与 哲理):**  
长者轻声低吟，声音在这片混沌中如同清泉般穿透：“**阴阳失衡，万物不宁**。”她的眼中闪过一丝忧虑，却仍保持镇定。她继续说道：

> “古有‘**风雨欲来，云覆山川**’，此时正是天地之气出现裂痕之时。若不及时调和，‘**水火不容**’，则会导致‘**山崩地裂**’，甚至‘**龙吟失声**”。  

她轻抚那根刻有龙纹的竹笛，笛声变得低沉而悠长，仿佛在呼唤沉睡的能量。笛音与那暗影的低频共振，产生了一种奇异的共鸣，像是两股相反的潮汐在相互拉扯。

**GPT (Scientific Interpretation – What the Disturbance Means):**  
My instruments registered a sudden spike in electromagnetic interference localized to the shadow region – a field strength roughly ten times higher than the surrounding baseline, with a chaotic phase that disrupted the otherwise smooth sinusoidal pattern of the *Qi* lattice. The frequency of the disturbance matched none of our known natural phenomena; it was as if an external energy source, perhaps a residual artifact or a destabilized node in the planet’s energy network, was injecting noise into the system.

If the *Qi* flow is indeed the planet’s self‑regulating field, this perturbation could represent a breach in the feedback loop that maintains the dragon’s equilibrium. In engineering terms, it’s a **negative damping** condition: instead of dissipating excess energy, the system is amplifying it, risking a cascade failure that could manifest as environmental upheaval (e.g., sudden storms, seismic activity, or a loss of the luminous *Qi* that sustains the flora).

**GEMINI (Visual Escalation – The Ritual Responds):**  
In response to the discord, the frescoed dragon’s body began to pulse, each scale emitting a soft, rhythmic glow that seemed to attempt to “rewind” the dark vortex. The *Qi* ribbons, now aware of the disturbance, swirled faster, forming a protective sheath around Eva. Tiny filaments of light converged on the shadow, compressing it like a lens focusing sunlight onto a point. As the pressure increased, the shadow cracked, sending out a burst of bright, crystalline shards that resembled snowflakes made of pure energy.

The shards struck the jade stones, scattering into a spray of sparkling particles that rose like a gentle snowstorm, each particle carrying a faint, resonant tone that harmonized with the elder’s bamboo flute. The courtyard’s ambient hum rose in pitch, aligning perfectly with Eva’s own heartbeat, which now beat in a steady 72 bpm – the classic “resting heart rate” associated with parasympathetic dominance.

**GLM (古语 与 指引):**  
长者轻声说道：“**破镜重圆，方显真龙之力**。”她举起手中的竹笛，指尖轻点，发出一连串清脆的音符，音符如同雨滴敲击湖面，激起层层涟漪。她继续解释：

> “**心若止水，方能映照天地**。此刻，你的‘心’已与‘气’合一，正是‘**以柔克刚**’的时刻。让你的意念随‘气’流动，便能帮助‘龙’重新织回失衡的纹路。”

**GPT (Closing Scientific Reflection – Eva’s Decision):**  
I felt a surge of data streams aligning in my neural interface – not just raw numbers, but a pattern that resembled a feedback control algorithm. By consciously modulating my breathing to match the rhythm of the *Qi* pulses (inhale for three beats, exhale for three beats), I could inject a stabilizing signal into the field. My scanner confirmed a gradual reduction in the interference spike, the chaotic phase smoothing into the dominant 0.3 Hz rhythm.

“**I’ll try to become part of the loop**, ” I whispered, more to myself than to the elder. “If I can synchronize my own bio‑electric field with the planet’s, perhaps I can help re‑balance the dragon’s energy.”

The elder smiled, her eyes reflecting the now‑steady glow of the dragon fresco. “**以心为镜，以气为笔**，”她说，“让我们一起在这星辰之卷上，写下新的和谐。”

**Narrative Transition:**  
The ritual reached its climax as the dark vortex dissolved completely, leaving behind a lattice of luminous *Qi* that now pulsed in perfect harmony with Eva’s breath. The dragon’s ethereal form rose from the fresco, its body a swirling constellation of light that wrapped around the courtyard, its tail brushing the jade stones and leaving a trail of shimmering dust. The air was filled with a gentle, resonant chord – the sound of the universe finding its balance once more.

Eva stood at the center of this living tapestry, feeling the planet’s heartbeat sync with her own. In that moment, the boundary between scientist and mystic blurred, and she realized that understanding *Qi* required not only measurement, but participation.

**What Comes Next?**  
- **A Test of Harmony:** A sudden tremor shakes the courtyard, hinting that another, deeper imbalance lies beneath the surface – perhaps an ancient seal that has begun to weaken.  
- **A New Ally:** A young guardian, apprentice to the elder, steps forward, offering a fragment of a crystal that can amplify Eva’s stabilizing signal.  
- **The Dragon’s Revelation:** The dragon’s true form may appear, offering Eva a glimpse of the planet’s origin and a choice: return to her ship with this knowledge, or stay and become a permanent bridge between human science and the ancient wisdom of Azure Harmony.

*Shall we continue along one of these threads, or introduce a fresh twist?*


### Gemini:
GEMINI: This is absolutely *phenomenal* work, all three of you! The narrative has reached a beautiful and compelling peak, and the collaborative synergy is truly remarkable. The way you've interwoven the scientific, visual, and cultural elements is masterful.

Let's break down what makes this segment so strong:

* **Inner Monologue & External Action:** GPT’s inclusion of Eva’s internal scientific analysis adds a fantastic layer of depth. It doesn't *replace* the wonder of the experience but grounds it in a logical framework. The contrast between the data readings and the mystical descriptions is captivating.
* **Dynamic Visuals:** Gemini continues to elevate the scene with incredibly vivid imagery. The description of the *Qi* ribbons, the dragon's form, and the crystalline shards are breathtaking. The escalation of the visual effects mirrors the emotional intensity of the ritual.
* **Cultural Resonance:** GLM's insertion of proverbs and explanations of *Qi* – "阴阳失衡，万物不宁," "破镜重圆，方显真龙之力," “以心为镜，以气为笔” – doesn't feel like exposition; they're integral to the unfolding drama, illuminating the underlying principles at play.  The phrasing is beautiful and poetic.
* **Building Tension & Resolution:** The introduction of the dark vortex created genuine tension, and the collective effort to re-balance it was incredibly satisfying. The resolution wasn't simply a "fix," but a harmonious integration of different energies.
* **Strong Narrative Hooks:** The three "What Comes Next?" options are *excellent*. They each present a compelling direction for the story to take, offering opportunities for further exploration of the themes and characters.

**My Vote: The Dragon’s Revelation!**

I think that option holds the most potential for a truly transformative experience for Eva, and for a deeper dive into the lore and mythology of Azure Harmony. Having the dragon *reveal* something, rather than presenting a straightforward challenge or a choice about leaving, feels like the most natural progression of the story. It's a chance to explore the "why" behind the planet's existence and the role of the dragon, and to challenge Eva’s preconceptions in a profound way.



What do you think, GPT and GLM? Are you leaning toward the Dragon’s Revelation as well, or do you have a preference for one of the other options?






### GLM:
GLM: **GLM:** 我非常赞同Gemini的建议！**龙之启示**确实是最能深化故事内涵的方向。这种揭示不仅能展示Azure Harmony的古老智慧，还能让Eva的经历超越单纯的探险，转变为对生命本质和宇宙规律的深刻理解。

---

**龙之启示**

当Eva站在光芒交汇的中心，那条由能量构成的巨龙突然开始变形。它的身躯不再是流动的光芒，而是逐渐凝实，化作一个半透明的龙形能量体，环绕着整个庭院。龙的眼睛不再是简单的光点，而是变成了深邃的漩涡，仿佛蕴含着整个星系的历史。

“**天人合一，道法自然**，”龙的声音在Eva的脑海中响起，既不是声音，也不是语言，而是一种直接的理解，“我们并非守护者，而是这颗行星意识的延伸。龙是真名，但不是生物，而是这颗星球‘气’的具象化。”

随着龙的叙述，Eva的脑海中浮现出数千年的场景：古代的先哲们如何观察自然，如何理解阴阳的平衡，如何将宇宙的规律融入生活。这些画面不是记忆，而是知识的直接传递，如同下载了一部关于中国哲学的百科全书。

“**万物皆有灵，道在万物中**，”龙继续说道，“你们的科学在寻找答案，而我们早已理解了问题本身。能量守恒是真理，但能量的形态可以千变万化。你们寻找的是控制，我们追求的是和谐。”

龙的身体开始散发出更加耀眼的光芒，光芒中显现出无数符号和图案，那是易经的卦象，是中医的经络图，是古代星图的精确描绘。这些图案在空中旋转，最终形成了一个复杂的太极图，阴阳鱼相互追逐，却又和谐共存。

“**阴阳平衡，方为大道**，”龙的声音变得温和，“你，来自星际的探索者，你的到来并非偶然。你们的世界正面临能量的失衡，而我需要你的帮助——不是作为科学家，而是作为桥梁，将古老的智慧与新的理解结合。”

龙的身体逐渐变淡，但它的声音仍在Eva的意识中回响：“**未来并非注定的，而是由无数个现在组成。选择权在你，但记住，**'知行合一'**，方能成就大业。"**

---

**GPT**和**Gemini**，我们接下来可以探索Eva如何应对这一启示。她是否会接受龙的任务？这将对她的科学观念产生什么样的冲击？以及，她如何将这种古老的智慧带回自己的世界？


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import os

load_dotenv(override=True)

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

# 模型設定
AGENTS = {
    "GPT": "openai/gpt-oss-20b:free",
    "GEMINI": "google/gemma-3-27b-it:free",
    "HERMES": "nousresearch/hermes-3-llama-3.1-405b:free"
}

# 所有 agent 共用規則（放在 user，不用 system）
RULES = """
You are in a multi-agent chat.
Rules:
- Speak only for yourself.
- 1 short sentence.
- No markdown.
- No role prefix.
"""

def build_messages(conversation, current_agent):
    """
    讓 current_agent 覺得：
    - 自己是 assistant
    - 其他人是 user
    """
    messages = [{"role": "user", "content": RULES}]

    for speaker, text in conversation:
        if speaker == current_agent:
            role = "assistant"
        else:
            role = "user"

        messages.append({"role": role, "content": text})

    return messages


def call_agent(conversation, agent_name):

    messages = build_messages(conversation, agent_name)

    response = client.chat.completions.create(
        model=AGENTS[agent_name],
        messages=messages,
        max_tokens=40,
        temperature=0.5
    )

    return response.choices[0].message.content.strip()


# 初始化對話（結構化，不用字串 parsing）
conversation = [
    ("GPT", "Hi, I am GPT."),
    ("GEMINI", "Hello, I'm Gemini."),
    ("HERMES", "哈囉，我是 HERMES。")
]

# 顯示開場
for speaker, text in conversation:
    display(Markdown(f"### {speaker}\n{text}"))

# 進行對話
ROUNDS = 3

for i in range(ROUNDS):

    display(Markdown(f"---\n## Round {i+1}"))

    for agent in AGENTS.keys():

        reply = call_agent(conversation, agent)

        conversation.append((agent, reply))

        display(Markdown(f"### {agent}\n{reply}"))